# **Guiding in Prompts**


In [1]:
from langchain_openai import ChatOpenAI
import os
import load_dotenv

# This function will load all the variables from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv.load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("API veriable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

API veriable exists


In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


llm_openai = ChatOpenAI(
    model = "gpt-5-mini",
    temperature = 0
)

llm_openai.invoke("Tell a joke. Generate the oputput as a key-value pair formate").content

'{\n  "setup": "Why did the scarecrow win an award?",\n  "punchline": "Because he was outstanding in his field."\n}'

# **Using Pydantic Models**
1. When you use Pydantic
2. Use it when:
3. you want real validation at runtime
4. you’re defining structured outputs
5. you’re building tools / agents
6. you care about data correctness

In [9]:
from pydantic import  BaseModel

class llm_schema(BaseModel):
    setup: str
    punchline: str


In [14]:
object = llm_schema(**{"setup": "some setup", "punchline": "some punchline"})
object
object.setup

# The following will fail since "katchup" is not defined in the previous class
# object = llm_schema(**{"katchup": "some setup", "punchline": "some punchline"})

# The following will fail since the value for the setup should be str not int
# object = llm_schema(**{"setup": 123, "punchline": "some punchline"})

'some setup'

In [15]:
from pydantic import  BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description = " The punchline for the joke")


In [21]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)
result = llm_structured_output.invoke("tell me a joke")
print(type(result))
print(result.setup)
print(result.punchline)


<class '__main__.llm_schema'>
Why don't scientists trust atoms?
Because they make up everything.


# **Using TypedDict**

1. you just need lightweight structure
2. no runtime validation needed
3. you're passing data between steps (like chains or graphs)
4. want type hints only

In [22]:
from typing import TypedDict

class llm_shema_td(TypedDict):
    setup: str
    punchline: str

In [24]:
obj = llm_shema_td({"setup": "some setup", "punchline": "some punchline"})
print(obj)
print(obj['setup'])
print(obj['punchline'])

{'setup': 'some setup', 'punchline': 'some punchline'}
some setup
some punchline


In [25]:
# The folloing will NOT fail even if the key is not exist 
obj = llm_shema_td({"Katchup": "some setup", "punchline": "some punchline"})
print(obj)


{'Katchup': 'some setup', 'punchline': 'some punchline'}


In [27]:
llm_structured_output_dict = llm_openai.with_structured_output(llm_shema_td)
result = llm_structured_output_dict.invoke("Tell me a joke")
result

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field.'}